Import

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
import pandas as pd
import numpy as np

from src.config import (
    TRAINED_MODELS_DIR,
    PROCESSED_DIR,
    RANDOM_STATE,
    TEST_SIZE
)

In [5]:
from src.preprocessing import (
    extract_id,
    read_raman_csv,
    process_folder,
    merge_ground_truth
)

from src.training import (
    prepare_data,
    split_data,
    train_model,
    evaluate_model,
    run_pipeline
)

from sklearn.svm import SVC
from xgboost import XGBClassifier

In [6]:
training_folder = "../data/raw/training/18062026"

gt_path = "../data/raw/metadata/ground_truth.csv"

In [7]:
df_spectra = process_folder(
    training_folder
)

df_ml = merge_ground_truth(
    df_spectra,
    gt_path
)

print(df_ml.shape)

df_ml.head()

[INFO] max spectrum length = 1901
(30, 1910)


,ID,Component1,Component2,Component3,Component4,Component5,Component6,Primary component,filename,f1,...,f1892,f1893,f1894,f1895,f1896,f1897,f1898,f1899,f1900,f1901
0,2137,Paracetamol,,,,,,Paracetamol,Scan 02137(1)_Sample_26_06_18_15_03_15_024.csv,5860.403,...,631.0392,703.8133,746.3972,699.6145,570.8422,492.3011,541.4897,590.2776,516.2697,418.0595
1,2138,Paracetamol,Glucose,,,,,Paracetamol,Scan 02138(1)_Sample_26_06_18_15_02_47_160.csv,7052.055,...,539.7412,467.9242,414.8203,356.1947,289.4112,258.8734,287.8663,320.7649,302.1977,252.1132
2,2139,Paracetamol,Glucose,,,,,Paracetamol,Scan 02139(1)_Sample_26_06_18_15_02_05_729.csv,8953.075,...,545.0209,494.0753,507.4854,507.4097,459.5291,420.3565,433.8285,476.1982,511.7477,508.5796
3,2140,,Glucose,,,,,Glucose,Scan 02140(1)_Sample_26_06_18_15_01_50_038.csv,11255.250,...,392.4122,439.0474,667.8126,858.1470,877.5813,782.0821,645.9015,536.3041,501.0833,496.7264
4,2141,,,,,,,Glucose,Scan 02141(1)_Sample_26_06_18_15_01_25_903.csv,11563.950,...,767.9711,691.2987,553.7587,500.7181,584.4156,671.1409,657.0325,589.2746,540.5200,568.8936


In [8]:
processed_file = (
    "../data/processed/raman_data_gt.csv"
)

df_ml.to_csv(
    processed_file,
    index=False
)

print(
    f"Saved to {processed_file}"
)

Saved to ../data/processed/raman_data_gt.csv


In [9]:
X, y = prepare_data(
    df_ml
)

print("X shape:", X.shape)
print("y shape:", y.shape)

print(
    "Positive class:",
    y.sum()
)

[INFO] Using target column: Primary component
[INFO] Number of features: 1902
X shape: (30, 1902)
y shape: (30,)
Positive class: 13


In [10]:
X_train, X_test, y_train, y_test = split_data(
    X,
    y
)

print(
    X_train.shape,
    X_test.shape
)

(21, 1902) (9, 1902)


In [11]:
svm_model = SVC(
    kernel="rbf",
    probability=True
)

trained_model, scaler = train_model(
    svm_model,
    X_train,
    y_train,
    scale=True
)

In [12]:
cm = evaluate_model(
    trained_model,
    X_test,
    y_test,
    scaler
)


Confusion Matrix:
[[5 0]
 [1 3]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      1.00      0.91         5
           1       1.00      0.75      0.86         4

    accuracy                           0.89         9
   macro avg       0.92      0.88      0.88         9
weighted avg       0.91      0.89      0.89         9



In [13]:
pd.Series(y).value_counts()

0    17
1    13
Name: count, dtype: int64

In [14]:
X.max(axis=1)[:10]

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=float32)

In [22]:
X_train.mean(), X_train.std()

(np.float32(0.28167763), np.float32(0.20618026))

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

print("Mean:", X_train_scaled.mean())
print("Std:", X_train_scaled.std())

Mean: -6.1123786e-09
Std: 0.9997371


In [16]:
xgb_model = XGBClassifier(
    random_state=RANDOM_STATE,
    eval_metric="logloss"
)

xgb_model, scaler = train_model(
    xgb_model,
    X_train,
    y_train,
    scale=False
)

In [17]:
cm_xgb = evaluate_model(
    xgb_model,
    X_test,
    y_test,
    scaler=None
)


Confusion Matrix:
[[5 0]
 [0 4]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      1.00      1.00         4

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9



In [18]:
svm_model = SVC(
    kernel="rbf",
    probability=True
)

svm_trained, svm_cm = run_pipeline(
    df_ml,
    svm_model,
    scale=True,
    model_name="svm"
)


===== SVM =====
[INFO] Using target column: Primary component
[INFO] Number of features: 1902

Confusion Matrix:
[[5 0]
 [1 3]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      1.00      0.91         5
           1       1.00      0.75      0.86         4

    accuracy                           0.89         9
   macro avg       0.92      0.88      0.88         9
weighted avg       0.91      0.89      0.89         9

[INFO] Saved model to /home/oraymond/Raman_Project/models/trained/svm.pkl


In [19]:
TRAINED_MODELS_DIR

PosixPath('/home/oraymond/Raman_Project/models/trained')

In [20]:
list(TRAINED_MODELS_DIR.glob("*"))

[PosixPath('/home/oraymond/Raman_Project/models/trained/svm.pkl')]

In [21]:
xgb_model = XGBClassifier(
    random_state=RANDOM_STATE,
    eval_metric="logloss"
)

xgb_trained, xgb_cm = run_pipeline(
    df_ml,
    xgb_model,
    scale=False,
    model_name="xgb"
)


===== XGB =====
[INFO] Using target column: Primary component
[INFO] Number of features: 1902

Confusion Matrix:
[[5 0]
 [0 4]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      1.00      1.00         4

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9

[INFO] Saved model to /home/oraymond/Raman_Project/models/trained/xgb.pkl


In [22]:
list(TRAINED_MODELS_DIR.glob("*.pkl"))

[PosixPath('/home/oraymond/Raman_Project/models/trained/xgb.pkl'),
 PosixPath('/home/oraymond/Raman_Project/models/trained/svm.pkl')]

In [23]:
import pickle

with open(
    TRAINED_MODELS_DIR / "svm.pkl",
    "rb"
) as f:
    bundle = pickle.load(f)

bundle.keys()


dict_keys(['model', 'scaler'])

In [24]:
bundle["model"]
bundle["scaler"]

,copy,True
,with_mean,True
,with_std,True


In [25]:
print("Feature count:", X.shape[1])

Feature count: 1902
